In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mutual_info_score, accuracy_score

In [2]:
url = (
    "https://raw.githubusercontent.com/DataTalksClub/"
    "machine-learning-zoomcamp/main/cohorts/2026/data/"
    "course_lead_scoring_2026.csv"
)

df = pd.read_csv(url)

categorical = ["lead_source", "industry", "employment_status", "location"]
numerical = ["annual_income", "number_of_courses_viewed", "interaction_count",
    "lead_score"]

features = categorical + numerical

print("Missing values:")
print(df.isna().sum())

Missing values:
lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64


In [3]:
df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)

print("Q1:")
print(df["industry"].value_counts())
print("Mode:", df["industry"].mode()[0])

Q1:
industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64
Mode: technology


In [4]:
correlations = df[numerical].corr()

print("Q2: correlation matrix")
print(correlations)

pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]

for a, b in pairs:
    print(a, b, correlations.loc[a, b])


Q2: correlation matrix
                          annual_income  number_of_courses_viewed  \
annual_income                  1.000000                  0.161300   
number_of_courses_viewed       0.161300                  1.000000   
interaction_count              0.122842                  0.721609   
lead_score                     0.229496                  0.757204   

                          interaction_count  lead_score  
annual_income                      0.122842    0.229496  
number_of_courses_viewed           0.721609    0.757204  
interaction_count                  1.000000    0.915746  
lead_score                         0.915746    1.000000  
interaction_count lead_score 0.9157457980418697
number_of_courses_viewed lead_score 0.7572042392104886
number_of_courses_viewed interaction_count 0.7216090038937035
annual_income interaction_count 0.12284235135955959


In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [6]:
y_train = df_train.pop("converted").values
y_val = df_val.pop("converted").values
y_test = df_test.pop("converted").values

y_full_train = df_full_train.pop("converted").values

In [9]:
mi = df_train[categorical].apply(
    lambda column: mutual_info_score(column, y_train)
)

print("Q3:")
print(mi.sort_values(ascending=False).apply(lambda score: round(score, 2)))

Q3:
lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64


In [11]:
def train_and_evaluate(selected_features, C=1.0):
    train_dicts = df_train[selected_features].to_dict(orient="records")
    val_dicts = df_val[selected_features].to_dict(orient="records")
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(train_dicts)
    X_val = dv.transform(val_dicts)

    model = LogisticRegression(
        solver="liblinear",
        C=C,
        max_iter=1000,
        random_state=42,
    )
    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_val)[:, 1]
    predictions = probabilities >= 0.5

    return accuracy_score(y_val, predictions)

In [12]:
baseline_accuracy = train_and_evaluate(features)

print("Q4:")
print("Original accuracy:", baseline_accuracy)
print("Rounded accuracy:", round(baseline_accuracy, 2))


Q4:
Original accuracy: 0.645
Rounded accuracy: 0.65


In [13]:
elimination_results = []

for feature in features:
    selected_features = [
        name for name in features if name != feature
    ]

    accuracy_without = train_and_evaluate(selected_features)
    difference = baseline_accuracy - accuracy_without

    elimination_results.append({
        "removed_feature": feature,
        "accuracy_without": accuracy_without,
        "difference": difference,
    })

elimination_results = pd.DataFrame(elimination_results)

print("Q5: all features")
print(elimination_results.to_string(index=False))

Q5: all features
         removed_feature  accuracy_without  difference
             lead_source             0.642       0.003
                industry             0.645       0.000
       employment_status             0.645       0.000
                location             0.645       0.000
           annual_income             0.724      -0.079
number_of_courses_viewed             0.643       0.002
       interaction_count             0.601       0.044
              lead_score             0.644       0.001


In [14]:
q5_options = [
    "lead_source",
    "number_of_courses_viewed",
    "interaction_count",
]

candidates = elimination_results[
    elimination_results["removed_feature"].isin(q5_options)
]

best_row = candidates.loc[candidates["difference"].idxmin()]
print("Q5 answer:", best_row["removed_feature"])


Q5 answer: number_of_courses_viewed


In [15]:
regularization_results = []

for C in [0.000001, 0.00001, 0.0001, 0.001]:
    accuracy = train_and_evaluate(features, C=C)
    rounded_accuracy = round(accuracy, 3)

    regularization_results.append((C, rounded_accuracy))
    print("Q6:", C, rounded_accuracy)

Q6: 1e-06 0.598
Q6: 1e-05 0.598
Q6: 0.0001 0.613
Q6: 0.001 0.645


In [16]:
best_C, best_accuracy = min(
    regularization_results,
    key=lambda result: (-result[1], result[0]),
)

print("Q6 answer:", best_C)

Q6 answer: 0.001
